# Motor de Búsqueda Booleano sobre un Corpus de Pokémon

**Asignatura:** Inteligencia Artificial — Recuperación de la Información  
**Referencia teórica:** *Introduction to Information Retrieval* (Manning, Raghavan & Schütze)  

En este cuaderno vamos a implementar un sistema de recuperación de información basado en el **modelo booleano** (IIR, Capítulo 1). Lo dividimos en cuatro partes:

1. **Normalización y Preprocesamiento** (IIR §2.2)  
2. **Construcción del Índice Invertido** (IIR §1.2)  
3. **Motor de Búsqueda Booleano** (IIR Cap. 1)  
4. **Evaluación del Sistema** (IIR §8.3)  

## 0. Instalación de dependencias

Ejecutar esta celda una vez para instalar las librerías que vamos a usar.

In [1]:
# Instalar nltk y whoosh (solo hace falta una vez)
%pip install nltk whoosh --quiet

Note: you may need to restart the kernel to use updated packages.


## 1. Normalización y Preprocesamiento (IIR §2.2)

Antes de construir el índice invertido hay que aplicar un pipeline de normalización lingüística al texto. Según la Sección 2.2 del IIR, esto reduce la variabilidad léxica del vocabulario y mejora la eficacia de la recuperación.

Nuestro pipeline:

1. **Tokenización:** segmentar el texto en tokens individuales.  
2. **Conversión a minúsculas (*case folding*):** para que `"Pikachu"` y `"pikachu"` sean el mismo término.  
3. **Eliminación de stopwords:** quitar palabras vacías (artículos, preposiciones...) con la lista en español de NLTK.  
4. **Stemming:** reducir cada token a su raíz morfológica con el *Snowball Stemmer* (ej: `"atacando"` → `"atac"`).  

Como el corpus viene de Smogon y está en inglés, combinamos las stopwords en español (requisito del enunciado) con las de inglés para que el filtrado sea más completo.

In [2]:
# Importaciones y descarga de recursos de NLTK
import nltk

# punkt_tab: tokenizador | stopwords: listas de palabras vacías
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

print("Recursos NLTK descargados correctamente.")

Recursos NLTK descargados correctamente.


In [3]:
# Configuración de stopwords y stemmer
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from nltk.tokenize import word_tokenize
import re

# Combinamos stopwords en español e inglés
# IIR §2.2.2: "The general trend [...] has been to use stopword removal."
stopwords_es = set(stopwords.words('spanish'))
stopwords_en = set(stopwords.words('english'))
STOPWORDS = stopwords_es | stopwords_en

print(f"Total de stopwords cargadas: {len(STOPWORDS)}")
print(f"  - Español: {len(stopwords_es)} palabras")
print(f"  - Inglés:  {len(stopwords_en)} palabras")
print(f"  - Ejemplo (español): {list(stopwords_es)[:10]}")
print(f"  - Ejemplo (inglés):  {list(stopwords_en)[:10]}")

# Usamos SnowballStemmer (Porter2) para inglés porque el corpus es en inglés
# IIR §2.2.4: "Stemming usually refers to a crude heuristic process that
#              chops off the ends of words."
stemmer = SnowballStemmer('english')

print(f"\nStemmer configurado: SnowballStemmer (english)")
print(f"  'attacking' -> '{stemmer.stem('attacking')}'")
print(f"  'effectiveness' -> '{stemmer.stem('effectiveness')}'")
print(f"  'resistances' -> '{stemmer.stem('resistances')}'")

Total de stopwords cargadas: 504
  - Español: 313 palabras
  - Inglés:  198 palabras
  - Ejemplo (español): ['como', 'o', 'fueses', 'por', 'tuviste', 'seríais', 'lo', 'mi', 'tuyas', 'sobre']
  - Ejemplo (inglés):  ['o', 'it', 'should', 'himself', 'ma', 'they', 'was', 'a', "mightn't", 'not']

Stemmer configurado: SnowballStemmer (english)
  'attacking' -> 'attack'
  'effectiveness' -> 'effect'
  'resistances' -> 'resist'


In [4]:
# Función de preprocesamiento completa

def preprocesar_texto(texto):
    """
    Aplica todo el pipeline de normalización a un texto:
    minúsculas -> limpiar caracteres raros -> tokenizar -> quitar stopwords -> stemming
    """
    # 1. Pasar a minúsculas
    texto = texto.lower()

    # 2. Quitar caracteres especiales (dejamos letras, números, espacios y guiones
    #    porque hay nombres como "iron-hands")
    texto = re.sub(r'[^a-záéíóúüñ0-9\s\-]', ' ', texto)

    # 3. Tokenizar
    tokens = word_tokenize(texto)

    # 4. Filtrar stopwords y tokens muy cortos
    tokens_filtrados = [t for t in tokens if t not in STOPWORDS and len(t) > 1]

    # 5. Aplicar stemming
    tokens_stem = [stemmer.stem(t) for t in tokens_filtrados]

    return ' '.join(tokens_stem)


# Probamos con un ejemplo
ejemplo = "Dragonite is a powerful Dragon-type attacker with great resistances and coverage moves."
print("Texto original:")
print(f"  {ejemplo}")
print("\nTexto preprocesado:")
print(f"  {preprocesar_texto(ejemplo)}")

Texto original:
  Dragonite is a powerful Dragon-type attacker with great resistances and coverage moves.

Texto preprocesado:
  dragonit power dragon-typ attack great resist coverag move


## 2. Construcción del Índice Invertido (IIR §1.2)

El índice invertido es la estructura de datos clave. Como dice el IIR (§1.2):

> *"For each term t, we have a list of all documents that contain t."*

Usamos **Whoosh** para construirlo. Es una librería de indexación y búsqueda de texto completo escrita en Python puro.

El esquema (`Schema`) tiene dos campos:
- **`titulo`** (`ID`): nombre del archivo, nos sirve como identificador del documento. Lo guardamos (`stored=True`) para poder mostrarlo en los resultados.
- **`contenido`** (`TEXT`): el texto del documento, al que le aplicamos un analizador personalizado con nuestro pipeline de stopwords + stemming.

In [5]:
# Definimos el esquema de Whoosh con nuestro analizador
import os
from whoosh import index
from whoosh.fields import Schema, TEXT, ID
from whoosh.analysis import RegexTokenizer, LowercaseFilter, StopFilter, StemFilter

# Analizador personalizado: replica el pipeline NLTK pero dentro de Whoosh
# RegexTokenizer -> minúsculas -> quitar stopwords -> stemming
analizador_personalizado = (
    RegexTokenizer(r'[\w\-]+')          # tokenizar (incluyendo guiones)
    | LowercaseFilter()                  # minúsculas
    | StopFilter(stoplist=STOPWORDS)      # stopwords ES + EN
    | StemFilter(stemfn=stemmer.stem)     # stemming con SnowballStemmer
)

# Definimos el schema
esquema = Schema(
    titulo=ID(stored=True, unique=True),
    contenido=TEXT(analyzer=analizador_personalizado, stored=True)
)

print("Esquema Whoosh definido:")
print(f"  Campos: {list(esquema.names())}")
print(f"  Analizador: RegexTokenizer -> LowercaseFilter -> StopFilter -> StemFilter")

Esquema Whoosh definido:
  Campos: ['contenido', 'titulo']
  Analizador: RegexTokenizer -> LowercaseFilter -> StopFilter -> StemFilter


In [6]:
# Función para construir el índice invertido a partir del corpus
import shutil

DIRECTORIO_INDICE = "../indice_booleano"
DIRECTORIO_CORPUS = "../corpus"


def construir_indice(directorio_corpus, directorio_indice, esquema):
    """
    Recorre la carpeta del corpus, lee cada .txt y mete todo en el índice de Whoosh.
    Si ya existía el directorio del índice, lo borra y lo recrea.
    """
    # Limpiar índice anterior si existe
    if os.path.exists(directorio_indice):
        shutil.rmtree(directorio_indice)
    os.makedirs(directorio_indice)

    ix = index.create_in(directorio_indice, esquema)
    writer = ix.writer()

    docs_indexados = 0
    docs_error = 0

    archivos = sorted([f for f in os.listdir(directorio_corpus) if f.endswith('.txt')])

    for nombre_archivo in archivos:
        ruta_completa = os.path.join(directorio_corpus, nombre_archivo)
        try:
            with open(ruta_completa, 'r', encoding='utf-8') as f:
                contenido = f.read()

            # Whoosh aplica el analizador automáticamente al indexar
            writer.add_document(
                titulo=nombre_archivo,
                contenido=contenido
            )
            docs_indexados += 1

        except Exception as e:
            print(f"  [ERROR] No se pudo indexar '{nombre_archivo}': {e}")
            docs_error += 1

    writer.commit()

    print(f"Índice construido en '{directorio_indice}'.")
    print(f"  Documentos indexados: {docs_indexados}")
    if docs_error > 0:
        print(f"  Documentos con error: {docs_error}")

    return ix

In [7]:
# Construimos el índice
ix = construir_indice(DIRECTORIO_CORPUS, DIRECTORIO_INDICE, esquema)

# Estadísticas básicas
with ix.reader() as reader:
    print(f"\nEstadísticas del índice:")
    print(f"  Nº de documentos: {reader.doc_count()}")
    print(f"  Tamaño del vocabulario: {reader.field_length('contenido')}")

Índice construido en '../indice_booleano'.
  Documentos indexados: 50

Estadísticas del índice:
  Nº de documentos: 50
  Tamaño del vocabulario: 274879


In [8]:
# Echamos un vistazo a los primeros 20 términos del índice para comprobar que
# el preprocesamiento ha funcionado bien

with ix.reader() as reader:
    print("Muestra de términos del índice invertido:")
    print(f"{'Término':<25} {'Nº Docs':>10} {'Frecuencia':>12}")
    print("-" * 50)

    contador = 0
    for term in reader.field_terms('contenido'):
        doc_freq = reader.doc_frequency('contenido', term)
        freq = reader.frequency('contenido', term)
        print(f"  {term:<25} {doc_freq:>8} {freq:>12}")
        contador += 1
        if contador >= 20:
            break

    total_terms = len(list(reader.field_terms('contenido')))
    print(f"\n  ... (mostrando 20 de {total_terms} términos)")

Muestra de términos del índice invertido:
Término                      Nº Docs   Frecuencia
--------------------------------------------------
  -1                              11         14.0
  -2                               2          2.0
  -ate                             1          1.0
  -ghost                           1          1.0
  -immun                           1          1.0
  -the                             2          2.0
  1-on-1                           1          2.0
  10                               1          1.0
  10-11                            2          2.0
  10-12                            1          1.0
  100                             10         15.0
  100-base                         1          1.0
  101                              1          1.0
  102                              1          1.0
  104                              2          3.0
  106                              1          2.0
  108                             13         19.0
  11-12

## 3. Motor de Búsqueda Booleano (IIR Capítulo 1)

El modelo booleano es el más básico de RI. Del IIR (Cap. 1):

> *"A document either matches or does not match a query [...] The Boolean retrieval model is a model for information retrieval in which we can pose any query which is in the form of a Boolean expression of terms."*

Operadores que soportamos:

| Operador | Significado | Ejemplo |
|----------|-------------|---------|
| `AND`    | Intersección: ambos términos deben aparecer | `fire AND dragon` |
| `OR`     | Unión: al menos uno de los términos | `water OR ice` |
| `NOT`    | Complemento: excluir un término | `steel AND NOT fairy` |

Whoosh ya trae soporte nativo para parsear consultas booleanas con `QueryParser`.

In [9]:
# Función principal de búsqueda booleana
from whoosh.qparser import QueryParser, AndGroup


def busqueda_booleana(indice, consulta_str, campo='contenido'):
    """
    Ejecuta una consulta booleana sobre el índice. Devuelve los documentos
    que satisfacen la expresión (AND, OR, NOT). Sin ranking, solo match/no match.
    """
    # Por defecto los términos sueltos se unen con AND
    parser = QueryParser(campo, schema=indice.schema, group=AndGroup)
    consulta = parser.parse(consulta_str)

    with indice.searcher() as searcher:
        # limit=None para devolver todos los que coincidan (modelo booleano puro)
        resultados = searcher.search(consulta, limit=None)
        documentos_recuperados = [hit['titulo'] for hit in resultados]

    return documentos_recuperados


print("Función de búsqueda booleana lista.")

Función de búsqueda booleana lista.


In [10]:
# Función para mostrar resultados de forma legible

def mostrar_resultados(consulta_str, resultados):
    """Imprime los resultados de una consulta de forma formateada."""
    print(f"\n{'=' * 60}")
    print(f" Consulta: {consulta_str}")
    print(f"{'=' * 60}")

    if resultados:
        print(f" Se encontraron {len(resultados)} documento(s):\n")
        for i, doc in enumerate(sorted(resultados), 1):
            print(f"   {i:>3}. {doc.replace('.txt', '')}")
    else:
        print(" No se encontraron documentos que satisfagan la consulta.")

    print(f"{'=' * 60}")

In [11]:
# Probamos con varias consultas de ejemplo para ver que funciona

consultas_ejemplo = [
    "fire AND dragon",                          # AND
    "\"stealth rock\" OR spikes",                # OR + frase exacta
    "sweeper AND NOT fairy",                     # NOT
    "water AND defensive AND NOT ice",           # combinación
    "\"trick room\"",                             # frase exacta
]

print("Probando consultas de ejemplo...")

for consulta in consultas_ejemplo:
    resultados = busqueda_booleana(ix, consulta)
    mostrar_resultados(consulta, resultados)

Probando consultas de ejemplo...

 Consulta: fire AND dragon
 Se encontraron 45 documento(s):

     1. amoonguss
     2. baxcalibur
     3. ceruledge
     4. cinderace
     5. clodsire
     6. corviknight
     7. darkrai
     8. deoxys-speed
     9. dondozo
    10. dragapult
    11. dragonite
    12. garchomp
    13. garganacl
    14. gholdengo
    15. gliscor
    16. gouging-fire
    17. great-tusk
    18. greninja
    19. hatterene
    20. heatran
    21. iron-boulder
    22. iron-hands
    23. iron-moth
    24. iron-treads
    25. iron-valiant
    26. kingambit
    27. kyurem
    28. landorus-therian
    29. latios
    30. meowscarada
    31. ogerpon-wellspring
    32. primarina
    33. raging-bolt
    34. rillaboom
    35. roaring-moon
    36. rotom-wash
    37. scizor
    38. skeledirge
    39. sneasler
    40. ting-lu
    41. toxapex
    42. volcarona
    43. walking-wake
    44. zamazenta
    45. zapdos

 Consulta: "stealth rock" OR spikes
 Se encontraron 50 documento(s):

     

In [12]:
# Interfaz interactiva por consola
# Descomentar la llamada del final para usarla

def interfaz_busqueda(indice):
    """
    Interfaz sencilla de texto para probar consultas booleanas.
    Escribir 'salir' o 'exit' para terminar.
    """
    print("\n" + "#" * 55)
    print("#  MOTOR DE BÚSQUEDA BOOLEANO - Corpus Pokémon")
    print("#  Operadores:  AND  |  OR  |  NOT")
    print("#  Ejemplo:     fire AND dragon AND NOT fairy")
    print("#  Salir:       escribir 'salir' o 'exit'")
    print("#" * 55)

    while True:
        print()
        consulta_str = input("Consulta > ").strip()

        if not consulta_str:
            print("  (Consulta vacía, escribe algo o 'salir')")
            continue

        if consulta_str.lower() in ('salir', 'exit', 'quit'):
            print("\n  Hasta la próxima!")
            break

        try:
            resultados = busqueda_booleana(indice, consulta_str)
            mostrar_resultados(consulta_str, resultados)
        except Exception as e:
            print(f"  Error al procesar la consulta: {e}")
            print("  Revisa la sintaxis (AND, OR, NOT).")


# Descomentar para activar la interfaz:
# interfaz_busqueda(ix)

Para usar la interfaz interactiva, descomenta `interfaz_busqueda(ix)` en la celda de arriba y ejecútala. Escribe `salir` cuando termines.

## 4. Evaluación del Sistema (IIR §8.3)

Para evaluar la calidad del motor de búsqueda, el IIR (§8.3) define dos métricas principales:

**Precisión (*Precision*):** fracción de documentos recuperados que son relevantes.

$$
\text{Precision} = P = \frac{|\text{Recuperados} \cap \text{Relevantes}|}{|\text{Recuperados}|}
$$

**Sensibilidad (*Recall*):** fracción de documentos relevantes que fueron recuperados.

$$
\text{Recall} = R = \frac{|\text{Recuperados} \cap \text{Relevantes}|}{|\text{Relevantes}|}
$$

Los documentos **relevantes** vienen dados por un Gold Standard (juicio humano) que tenemos en el archivo `necesidades_info/necesidades.csv`.

In [13]:
# Función de evaluación: calcula Precision y Recall

def evaluar_rendimiento(recuperados, relevantes):
    """
    Calcula Precision y Recall comparando los documentos recuperados
    contra el Gold Standard (relevantes).

    Devuelve un dict con precision, recall, TP, FP y FN.
    """
    set_recuperados = set(recuperados)
    set_relevantes = set(relevantes)

    # TP: recuperados que sí son relevantes
    verdaderos_positivos = set_recuperados & set_relevantes
    # FP: recuperados que no son relevantes
    falsos_positivos = set_recuperados - set_relevantes
    # FN: relevantes que no se recuperaron
    falsos_negativos = set_relevantes - set_recuperados

    # Precision: TP / total recuperados (si no hay recuperados -> 0)
    if len(set_recuperados) == 0:
        precision = 0.0
    else:
        precision = len(verdaderos_positivos) / len(set_recuperados)

    # Recall: TP / total relevantes (si no hay relevantes -> 0)
    if len(set_relevantes) == 0:
        recall = 0.0
    else:
        recall = len(verdaderos_positivos) / len(set_relevantes)

    return {
        'precision': precision,
        'recall': recall,
        'verdaderos_positivos': verdaderos_positivos,
        'falsos_positivos': falsos_positivos,
        'falsos_negativos': falsos_negativos
    }


print("Función de evaluación lista.")

Función de evaluación lista.


In [14]:
# Función para imprimir la evaluación detallada de una consulta

def mostrar_evaluacion(consulta_str, descripcion, recuperados, relevantes):
    """Evalúa y muestra Precision/Recall con detalle de TP, FP, FN."""
    metricas = evaluar_rendimiento(recuperados, relevantes)

    print(f"\n{'=' * 65}")
    print(f" Necesidad: {descripcion}")
    print(f" Consulta:  {consulta_str}")
    print(f"{'=' * 65}")

    print(f"\n  Recuperados: {len(recuperados)}")
    print(f"  Relevantes (Gold Standard): {len(relevantes)}")
    print(f"  TP: {len(metricas['verdaderos_positivos'])}  |  FP: {len(metricas['falsos_positivos'])}  |  FN: {len(metricas['falsos_negativos'])}")
    print(f"\n  Precision: {metricas['precision']:.4f}")
    print(f"  Recall:    {metricas['recall']:.4f}")

    if metricas['verdaderos_positivos']:
        print(f"\n  TP (recuperados y relevantes):")
        for doc in sorted(metricas['verdaderos_positivos']):
            print(f"      - {doc}")

    if metricas['falsos_positivos']:
        print(f"\n  FP (recuperados pero no relevantes):")
        for doc in sorted(metricas['falsos_positivos']):
            print(f"      - {doc}")

    if metricas['falsos_negativos']:
        print(f"\n  FN (relevantes pero no recuperados):")
        for doc in sorted(metricas['falsos_negativos']):
            print(f"      - {doc}")

    print(f"{'=' * 65}")

    return metricas

In [15]:
# Cargamos las necesidades de información desde el CSV
import csv

RUTA_NECESIDADES = "../necesidades_info/necesidades.csv"

def cargar_necesidades(ruta_csv):
    """
    Lee el CSV de necesidades de información.
    Cada fila tiene: id, descripcion, consulta_booleana, consulta_texto_libre, relevantes
    Los nombres de los relevantes vienen sin .txt, así que se lo añadimos.
    """
    necesidades = []

    with open(ruta_csv, 'r', encoding='utf-8') as f:
        lector = csv.DictReader(f)
        for fila in lector:
            # Convertir la columna de relevantes (string separado por comas) a lista
            docs_relevantes_raw = fila['relevantes'].strip()
            docs_relevantes = [
                nombre.strip() + '.txt'
                for nombre in docs_relevantes_raw.split(',')
                if nombre.strip()
            ]

            necesidades.append({
                'id': int(fila['id']),
                'descripcion': fila['descripcion'].strip(),
                'consulta_booleana': fila['consulta_booleana'].strip(),
                'consulta_texto_libre': fila['consulta_texto_libre'].strip(),
                'relevantes': docs_relevantes
            })

    return necesidades


necesidades = cargar_necesidades(RUTA_NECESIDADES)

print(f"Cargadas {len(necesidades)} necesidades de información.")
print(f"\nPrimeras 3:")
for n in necesidades[:3]:
    print(f"\n  [{n['id']}] {n['descripcion'][:80]}...")
    print(f"      Consulta: {n['consulta_booleana']}")
    print(f"      Relevantes: {[d.replace('.txt','') for d in n['relevantes']]}")

Cargadas 20 necesidades de información.

Primeras 3:

  [1] Pokémon ofensivo físico con Danza Espada y un ataque de prioridad con STAB para ...
      Consulta: Swords Dance AND (priority OR "Bullet Punch" OR "Sucker Punch")
      Relevantes: ['kingambit', 'scizor', 'ceruledge']

  [2] Pivot defensivo que pueda quitar hazards del campo (con Defog o Rapid Spin) y qu...
      Consulta: ("Defog" OR "Rapid Spin") AND pivot AND (resist OR immune)
      Relevantes: ['corviknight', 'zapdos', 'great-tusk']

  [3] Atacante especial muy rápido que sea resistente o inmune al tipo Hada para frena...
      Consulta: fast AND special AND (Fairy AND resist)
      Relevantes: ['gholdengo', 'iron-moth']


In [16]:
# Evaluamos el motor booleano con las 20 necesidades

print("Evaluación del motor booleano sobre las 20 necesidades de información:")
print()

resultados_evaluacion = []

for n in necesidades:
    recuperados = busqueda_booleana(ix, n['consulta_booleana'])

    metricas = mostrar_evaluacion(
        consulta_str=n['consulta_booleana'],
        descripcion=f"[{n['id']}] {n['descripcion']}",
        recuperados=recuperados,
        relevantes=n['relevantes']
    )

    resultados_evaluacion.append({
        'id': n['id'],
        'descripcion': n['descripcion'],
        'consulta': n['consulta_booleana'],
        'precision': metricas['precision'],
        'recall': metricas['recall'],
        'n_recuperados': len(recuperados),
        'n_relevantes': len(n['relevantes']),
        'tp': len(metricas['verdaderos_positivos']),
    })

Evaluación del motor booleano sobre las 20 necesidades de información:


 Necesidad: [1] Pokémon ofensivo físico con Danza Espada y un ataque de prioridad con STAB para limpiar equipos.
 Consulta:  Swords Dance AND (priority OR "Bullet Punch" OR "Sucker Punch")

  Recuperados: 39
  Relevantes (Gold Standard): 3
  TP: 3  |  FP: 36  |  FN: 0

  Precision: 0.0769
  Recall:    1.0000

  TP (recuperados y relevantes):
      - ceruledge.txt
      - kingambit.txt
      - scizor.txt

  FP (recuperados pero no relevantes):
      - baxcalibur.txt
      - cinderace.txt
      - clefable.txt
      - clodsire.txt
      - corviknight.txt
      - darkrai.txt
      - deoxys-speed.txt
      - dragapult.txt
      - dragonite.txt
      - garchomp.txt
      - garganacl.txt
      - gholdengo.txt
      - gliscor.txt
      - great-tusk.txt
      - greninja.txt
      - heatran.txt
      - iron-boulder.txt
      - iron-hands.txt
      - iron-moth.txt
      - iron-treads.txt
      - iron-valiant.txt
      - kyur

In [17]:
# Tabla resumen con todas las evaluaciones

print("\n" + "=" * 90)
print(" RESUMEN - Evaluación del motor booleano")
print("=" * 90)

print(f"\n{'ID':<5} {'Consulta':<45} {'Recup.':>7} {'Relev.':>7} {'TP':>5} {'Prec.':>8} {'Recall':>8}")
print("-" * 90)

for r in resultados_evaluacion:
    consulta_corta = r['consulta'][:43] + '...' if len(r['consulta']) > 45 else r['consulta']
    print(f"  {r['id']:<3} {consulta_corta:<45} {r['n_recuperados']:>5} {r['n_relevantes']:>7} {r['tp']:>5} {r['precision']:>8.4f} {r['recall']:>8.4f}")

# Promedios (macro-average)
precision_media = sum(r['precision'] for r in resultados_evaluacion) / len(resultados_evaluacion)
recall_medio = sum(r['recall'] for r in resultados_evaluacion) / len(resultados_evaluacion)

print("-" * 90)
print(f"  {'PROMEDIO (macro-average)':<53} {'':>5} {'':>7} {'':>5} {precision_media:>8.4f} {recall_medio:>8.4f}")
print("=" * 90)

print(f"\n  Precisión media:  {precision_media:.4f} ({precision_media*100:.1f}%)")
print(f"  Recall medio:     {recall_medio:.4f} ({recall_medio*100:.1f}%)")


 RESUMEN - Evaluación del motor booleano

ID    Consulta                                       Recup.  Relev.    TP    Prec.   Recall
------------------------------------------------------------------------------------------
  1   Swords Dance AND (priority OR "Bullet Punch...    39       3     3   0.0769   1.0000
  2   ("Defog" OR "Rapid Spin") AND pivot AND (re...    45       3     3   0.0667   1.0000
  3   fast AND special AND (Fairy AND resist)          45       2     2   0.0444   1.0000
  4   Stealth Rock AND Ground AND defensive            49       5     5   0.1020   1.0000
  5   Knock Off AND "Special Defense" AND wall         44       3     3   0.0682   1.0000
  6   Water AND defensive AND (recover OR synthes...    36       3     3   0.0833   1.0000
  7   Spore AND "Regenerator"                          10       1     1   0.1000   1.0000
  8   Taunt AND fast AND physical AND (Dark OR Fi...    39       2     2   0.0513   1.0000
  9   Magic Bounce AND special                    

## Conclusiones

Se ha implementado un sistema de recuperación de información basado en el modelo booleano siguiendo el IIR:

1. **Preprocesamiento (IIR §2.2):** pipeline con NLTK que incluye tokenización, case folding, eliminación de stopwords en español e inglés, y stemming con SnowballStemmer.

2. **Índice invertido (IIR §1.2):** construido con Whoosh sobre los 50 documentos del corpus, usando un analizador que integra el pipeline anterior.

3. **Motor booleano (IIR Cap. 1):** búsqueda con operadores AND, OR y NOT, más una interfaz interactiva de consola.

4. **Evaluación (IIR §8.3):** cálculo de Precisión y Recall para las 20 necesidades de información del CSV, comparando contra el Gold Standard.